# 03 · MemoryOS — 실제 대화로 동작 확인

MemoryOS에 실제 LoCoMo 대화를 넣고, 세그먼트 생성 → 검색 → 승격 →
eviction이 실제로 어떻게 일어나는지 단계별로 확인한다.

```
STM (큐 7칸)          최근 대화 + chain 요약
  ↓ 차면 FIFO 이관
MTM (세그먼트)        주제별 묶음, heat = 0.8·N_visit + 0.8·L + γ·recency
  ↓ heat > 5 승격        ↓ 자리 부족하면 heat 최저 삭제
LPM (영구)            사용자 프로필 + 사실들
```

**데이터 — conv-26의 세션 3·4만 사용한다.**
conv-26은 LoCoMo 대화 10편 중 첫 번째(Caroline & Melanie, 세션 19개
419발화 — 노트북 01·02에서 본 그 대화)다. 전체를 다 넣으면 API 비용이
크고 관찰할 것도 많아지므로 두 세션(41발화 ≈ 페이지 20개)만 쓴다.
이 조합을 고른 이유는 주제가 갈리기 때문 — S3(6월 9일)는 학교 연설
이벤트라는 하나의 화제, S4(6월 27일)는 목걸이 선물·결혼·캠핑 등 여러
근황. 서로 다른 주제가 서로 다른 세그먼트로 묶이는 걸 볼 수 있다.

**설정 — MTM 용량을 4칸(세그먼트 최대 4개)으로 줄인다.**
논문 기본값은 200칸인데, 이 조각에서는 세그먼트가 ~14개밖에 안 생겨서
기본값으로는 자리가 영원히 남는다 = eviction이 한 번도 일어나지 않는다.
삭제 정책을 관찰하려면 자리 경쟁을 일부러 만들어야 하므로 4칸으로 줄인다.
(승격·검색 등 나머지 동작은 논문 기본값 그대로.)

> 이 노트북은 실제 LLM(Groq)을 호출한다: **~110호출, ~10분, GROQ_API_KEY 필요.**

## 준비 — 부품 조립

용어 구분: **세션** = 데이터가 주는 단위(날짜 찍힌 하루치 대화),
**세그먼트** = 시스템(MTM)이 만드는 단위(주제가 비슷한 페이지 묶음).
한 세션이 여러 세그먼트로 흩어질 수도 있고, 다른 세션의 페이지가
기존 세그먼트에 합류할 수도 있다 (3단계에서 실제로 일어난다).

아래 셀이 하는 일:

1. conv-26을 로드하고 세션 3·4를 꺼낸다
2. LLM에 연결한다 (`llm.calls`/`llm.total_tokens`로 비용이 자동 집계됨).
   이 기록은 Groq로 실행됐다 — 현재 기본은 로컬 LM Studio(`default_provider()`)
3. MemoryOS를 만든다 — `on_evict=victims.append`는 "세그먼트가 지워질
   때마다 victims 리스트에 담아줘"라는 관찰용 콜백.
   eviction의 단위는 페이지가 아니라 **세그먼트(페이지 묶음)**다 —
   세그먼트가 지워지면 안에 든 페이지가 통째로 사라진다
4. 도우미 2개를 정의한다: `ingest_session`(세션의 발화를 순서대로 넣기),
   `memory_map`(지금 MTM에 어떤 세그먼트가 있는지 heat 순으로 출력)

In [1]:
from memlab.data import load_locomo
from memlab.evaluation import set_f1
from memlab.llm import GroqProvider
from memlab.methods import Utterance
from memlab.methods.memoryos import MemoryOS, MemoryOSConfig

sample = load_locomo()[0]
s3 = next(s for s in sample.sessions if s.index == 3)
s4 = next(s for s in sample.sessions if s.index == 4)

llm = GroqProvider()
victims = []  # eviction 관찰: 삭제되는 세그먼트가 여기 쌓인다
memoryos = MemoryOS(llm, sample.speaker_a, sample.speaker_b,
                    config=MemoryOSConfig(mtm_capacity=4),
                    on_evict=victims.append)

def ingest_session(session):
    for turn in session.turns:
        memoryos.ingest(Utterance(turn.speaker, turn.text,
                                 session.date_time, turn.blip_caption))

def memory_map(title):
    print(f"── {title} ──")
    for seg in sorted(memoryos.mtm.segments.values(), key=lambda g: -g.H_segment):
        print(f"  {len(seg.details)}p N={seg.N_visit} heat={seg.H_segment:.1f}"
              f"  {seg.summary[:58]}")

print(f"{sample.speaker_a} & {sample.speaker_b} / "
      f"S3 {len(s3.turns)}발화({s3.date_time}) + S4 {len(s4.turns)}발화({s4.date_time})")

Caroline & Melanie / S3 23발화(7:55 pm on 9 June, 2023) + S4 18발화(10:37 am on 27 June, 2023)


## 1. Ingest — 대화가 세그먼트로 묶인다

S3의 발화 23개를 순서대로 넣는다. 발화 하나가 들어올 때마다 내부에서
일어나는 일:

1. 발화 두 개(Caroline의 말 = Q, Melanie의 답 = R)가 **페이지** 하나로 묶인다
2. 페이지가 STM 큐(7칸)에 들어간다 — 이때 LLM이 직전 대화와 이어지는지
   판단하고, 이어지면 chain으로 연결해 흐름 요약(**meta_chain**)을 갱신한다
3. 큐가 차면 가장 오래된 페이지가 MTM으로 넘어간다 — 기존 세그먼트와
   주제가 비슷하면(F_score > 0.6) 합류하고, 아니면 새 세그먼트가 된다

출력 읽는 법: `1p N=0 heat=0.8` = 페이지 1개 / 검색된 적 0번 / heat 0.8

In [2]:
ingest_session(s3)
memoryos.flush_open_page()

memory_map("S3 ingest 후 MTM — 세그먼트별 (크기, 방문, heat, LLM이 쓴 요약)")

sample_page = memoryos.stm.get_all()[-1]
print(f"\nchain의 산물 — 한 페이지의 meta_chain (LLM이 쓴 실물):")
print(f"  {sample_page.meta_info}")
print(f"\n지금까지 {llm.calls}호출 / {llm.total_tokens:,}토큰")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

── S3 ingest 후 MTM — 세그먼트별 (크기, 방문, heat, LLM이 쓴 요약) ──
  1p N=0 heat=0.8  The conversation highlights the importance of sharing pers
  1p N=0 heat=0.8  Caroline expresses her commitment to using her voice to fo
  1p N=0 heat=0.8  The conversation focuses on mutual encouragement and share
  1p N=0 heat=0.8  The user expresses gratitude for the strong support and mo

chain의 산물 — 한 페이지의 meta_chain (LLM이 쓴 실물):
  Caroline shares her experience speaking at a school event about her transgender journey, receiving encouragement from Melanie. The conversation then shifts to personal life updates, where both friends discuss their support systems and share photos of their families and wedding.

지금까지 41호출 / 16,146토큰


출력에서 볼 것:

- **세그먼트가 4개뿐이고 전부 1페이지인 이유**: S3은 페이지 ~11개인데
  최근 7개는 아직 STM에 있고, 밀려난 4개만 MTM에 도착했다. 그런데 넷 다
  병합 기준(F_score > 0.6)을 못 넘어 각자 혼자가 됐다 — 실제 대화에서
  병합은 생각보다 드물다
- **heat가 전부 0.8인 이유**: heat ≈ 0.8×N_visit + 0.8×L. 검색된 적이
  없고(N=0) 페이지도 1개라(L=1) 0.8×1 = 0.8
- **meta_chain이 연설 외의 이야기까지 담는 이유**: LLM이 "대화가 계속
  이어진다"고 판단해 chain이 길게 유지됐고, 요약은 chain 전체를 덮는다

## 2. 검색 — 검색된 세그먼트는 heat가 오른다

`answer()`는 질문에 답하기 전에 MTM을 검색한다 (관련 세그먼트 top-5 →
그 안에서 관련 페이지 top-10). 논문 3.3: 이때 검색에 걸린 세그먼트는
N_visit이 1씩 오른다.

연설 이벤트에 대해 다섯 번 질문한다. 답도 받지만, 여기서 진짜 보려는 건
**질문할 때마다 관련 기억의 heat가 오르는 부수 효과**다.

In [3]:
rehearsal_questions = [
    "What event did Caroline speak at?",
    "How did Caroline feel about her school speech?",
    "Who supported Caroline at the event?",
    "What did Caroline talk about at the school event?",
    "What community was Caroline's speech about?",
]
for question in rehearsal_questions:
    answer = memoryos.answer(question)
    print(f"Q: {question}\nA: {answer}\n")

memory_map("검색 5회 후 — heat 변화")

Q: What event did Caroline speak at?
A: school event



Q: How did Caroline feel about her school speech?
A: grateful



Q: Who supported Caroline at the event?
A: friends, family and mentors



Q: What did Caroline talk about at the school event?
A: her transgender journey



Q: What community was Caroline's speech about?
A: transgender

── 검색 5회 후 — heat 변화 ──
  1p N=5 heat=4.8  The conversation highlights the importance of sharing pers
  1p N=5 heat=4.8  Caroline expresses her commitment to using her voice to fo
  1p N=5 heat=4.8  The conversation focuses on mutual encouragement and share
  1p N=5 heat=4.8  The user expresses gratitude for the strong support and mo


출력에서 볼 것:

- heat 0.8 → **4.8**: N_visit이 5가 되어 0.8×(5+1) = 4.8
- **네 세그먼트가 똑같이 오른 이유**: 지금 MTM엔 페이지가 4개뿐이라 어떤
  질문이든 전부 top-10 안에 걸린다. 기억이 많아지면 검색이 선별적이 되어
  heat에 차이가 생긴다
- 4.8은 승격 문턱(5)에 한 끗 모자란다 — 다음 단계에서 넘는다

## 3. 승격 — heat > 5 세그먼트는 LPM으로 넘어간다

S4를 이어서 넣는다. 페이지가 처리될 때마다 시스템이 "heat > 5인 세그먼트가
있나?"를 검사하고, 있으면 **승격**시킨다:

- 그 세그먼트의 페이지들을 LLM이 분석해 사용자 **프로필**을 갱신하고,
  **사실(fact)**을 뽑아 KB(User Knowledge Base, 100칸)에 저장한다
- 세그먼트 자체는 MTM에 남지만 L_interaction이 0으로 리셋된다
  (heat가 뚝 떨어져 곧바로 재승격되는 걸 막는 장치)

산수: 2단계에서 heat 4.8(N=5, L=1)인 세그먼트에 S4 페이지가 하나 합류하면
L=2가 되어 0.8×(5+2) = 5.6 > 5 → 승격.

In [4]:
kb_size_before = len(memoryos.lpm.user_kb)
ingest_session(s4)
memoryos.flush_open_page()

print(f"승격 발생: KB {kb_size_before}개 → {len(memoryos.lpm.user_kb)}개\n")
print("── LPM: User Profile (앞부분) ──")
print(memoryos.lpm.user_profile[:400])
print("\n── LPM: User KB (LLM이 추출한 사실들) ──")
for fact in memoryos.lpm.user_kb.all_texts():
    print(f"  - {fact}")

승격 발생: KB 0개 → 2개

── LPM: User Profile (앞부분) ──
Need for Belonging (High): The user explicitly states that friends, family, and mentors are their "rocks" and source of strength, emphasizing the critical role of their support system.
Family Concern (High): The user shares a photo of their family and discusses the importance of family and friends in their life, indicating a strong focus on familial and close social bonds.
Emotional Expression (Hi

── LPM: User KB (LLM이 추출한 사실들) ──
  - Moved from home country 4 years ago
  - Experienced breakup


출력에서 볼 것:

- **프로필의 형식** — "Need for Belonging (High): 근거…"는 90개 성격 차원에
  High/Medium/Low 등급을 매기는 프롬프트의 출력이다
- **KB의 사실 2개** — 승격된 세그먼트의 페이지들에서 추출됐다. 연설이 아니라
  이사·이별 이야기인 이유: 승격은 S4 페이지가 기존 세그먼트에 합류해 문턱을
  넘는 순간 일어났고, 그 시점의 세그먼트 내용이 분석 대상이었기 때문
- *"Moved from **home country**"* — 나라 이름이 사라진 걸 기억해 두자.
  5단계에서 문제가 된다

## 4. Eviction — 어떤 세그먼트가 지워지는가

MTM은 4칸인데 S4가 흐르는 동안 새 세그먼트가 계속 생겼다. **5개째가 되는
순간마다 heat가 가장 낮은 세그먼트가 지워진다** — 그 목록이 준비 단계에서
걸어둔 `on_evict` 콜백으로 `victims`에 쌓여 있다. 무엇이 지워졌는지 보면
이 시스템의 삭제 정책이 드러난다.

In [5]:
print(f"삭제된 세그먼트 {len(victims)}개 — (크기, 방문수, 내용):")
for victim in victims:
    print(f"  {len(victim.details)}p N={victim.N_visit}  {victim.summary[:55]}")

memory_map("\n생존자")

삭제된 세그먼트 10개 — (크기, 방문수, 내용):
  1p N=0  Caroline shares her positive experience leading a schoo
  1p N=0  Caroline shares the empowering experience of delivering
  1p N=0  The user compliments a wedding photo and asks about the
  1p N=0  Caroline congratulates Melanie on her wedding, and Mela
  1p N=0  A joyful family day spent playing games, sharing meals,
  1p N=0  Cherishing family moments
  1p N=0  The value of family and spending time with loved ones.
  1p N=0  Caroline shares a photo of a new necklace featuring a c
  1p N=0  Caroline shares the sentimental significance of a neckl
  1p N=0  The conversation covers Caroline sharing a sentimental 
── 
생존자 ──
  1p N=5 heat=4.8  The conversation highlights the importance of sharing pers
  1p N=5 heat=4.8  Caroline expresses her commitment to using her voice to fo
  1p N=5 heat=4.8  The conversation focuses on mutual encouragement and share
  2p N=5 heat=4.0  The conversation focuses on the importance of personal sup


지워진 것은 전부 **작고(1페이지) 한 번도 검색되지 않은 새 세그먼트**다.
heat = 0.8·N + 0.8·L에서 recency 항(γ=0.0001)이 사실상 꺼져 있어서,
새 세그먼트는 자리 잡기 전에 항상 최저 heat다.
**무엇을 왜 지우는지 묻지 않는 삭제** — 이 문제가 이후 우리 연구
(cause-aware forgetting)의 출발점이다.

## 5. QA — 기억의 상태가 점수를 가른다

상태가 다른 다섯 문제를 골라 답하게 하고, 노트북 02에서 만든 `set_f1`
(1.0 만점)로 채점한다. 기대: 남아 있는 기억은 맞히고, 지워진 기억은
못 맞힌다.

| 문제 | 근거가 되는 기억의 상태 |
|---|---|
| 연설 시기 | 검색 5회로 살아남음 (MTM) |
| 4년 전 어디서 이사왔나 | 두 세션을 종합해야 함 (multi-hop) |
| Mel 부부 결혼 몇 년 차 | **지워짐** (4단계 victims 목록에 있음) |
| 캠핑에서 뭐 했나 | 경계 사례 — S4 중반 |
| 어떤 공간을 만들고 싶나 | STM에 남은 마지막 화제 |

In [6]:
exam = [
    (8, "여러 번 검색된 연설 내용"),
    (11, "두 세션 종합 (multi-hop)"),
    (90, "삭제된 결혼 이야기"),
    (95, "경계 사례 — S4 중반의 캠핑"),
    (100, "STM에 남은 S4 마지막 화제"),
]
for index, fate in exam:
    qa = sample.qa[index]
    prediction = memoryos.answer(qa.question)
    print(f"[{fate}]")
    print(f"Q: {qa.question}")
    print(f"  정답: {qa.answer}")
    print(f"  예측: {prediction}")
    print(f"  set_f1 = {set_f1(prediction, qa.answer):.2f}\n")

print(f"총 비용: {llm.calls}호출 / {llm.total_tokens:,}토큰")

[여러 번 검색된 연설 내용]
Q: When did Caroline give a speech at a school?
  정답: The week before 9 June 2023
  예측: June 2023
  set_f1 = 0.50



[두 세션 종합 (multi-hop)]
Q: Where did Caroline move from 4 years ago?
  정답: Sweden
  예측: home country
  set_f1 = 0.00



[삭제된 결혼 이야기]
Q: How long have Mel and her husband been married?
  정답: Mel and her husband have been married for 5 years.
  예측: Information not provided
  set_f1 = 0.00



[경계 사례 — S4 중반의 캠핑]
Q: What did Melanie and her family do while camping?
  정답: explored nature, roasted marshmallows, and went on a hike
  예측: explored nature, roasted marshmallows around the campfire and went on a hike
  set_f1 = 0.86



[STM에 남은 S4 마지막 화제]
Q: What kind of place does Caroline want to create for people?
  정답: a safe and inviting place for people to grow
  예측: a safe, inviting place for people to grow
  set_f1 = 0.94

총 비용: 108호출 / 48,874토큰


점수가 기억의 상태를 그대로 따라간다 — 그리고 multi-hop 실패(0.00)의
원인이 흥미롭다. KB에 승격된 사실을 보면 *"Moved from **home country**
4 years ago"* — 지식 추출 과정에서 "Sweden"이 **일반적인 표현으로
뭉개졌다**. 원본 페이지(목걸이 이야기)는 이미 지워졌으니, 남은 건
뭉개진 요약뿐.

**eviction은 원본을 지우고, 요약은 세부를 지운다.** 둘이 겹치면 답할 수 없다.

## 정리

- **Ingest**: 대화가 chain 요약을 달고 주제별 세그먼트로 묶인다 (실제론 잘게 쪼개진다)
- **검색 → 승격**: 자주 검색된 세그먼트는 heat가 올라 LPM으로 넘어간다
- **Eviction**: 자리가 부족하면 heat가 가장 낮은 세그먼트가 지워진다 — 지워지는 건
  늘 작고 새로운 세그먼트다 (recency 항이 꺼져 있어서). 무엇을 왜 지우는지 묻지 않는다
- QA 점수는 기억의 상태를 그대로 따라간다

**다음 — 04 · 실험**: 대화 10편 전체를 돌려 baseline 수치를 얻는다
(러너, 체크포인트, 장시간 로컬 런 관리).